[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/mp-1/blob/master/notebooks/06_comparison.ipynb)

# 06. Сравнение методов

Сводим результаты всех методов в одну таблицу и строим общий график сходимости.

In [ ]:
!pip install -q numpy pandas sympy matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

Q = np.array([[4.0, -2.0], [-2.0, 6.0]])
B = np.array([-8.0, 6.0])
C = 30.0
X0 = np.array([0.0, 0.0])
EPS = 1e-4
X_STAR = np.linalg.solve(Q, -B)
F_STAR = 21.6

def f(point):
    x = np.asarray(point, dtype=float)
    return float(0.5 * x @ Q @ x + B @ x + C)

def grad(point):
    x = np.asarray(point, dtype=float)
    return Q @ x + B

def original_formula(x, y):
    return 2 * (x - 3) ** 2 + 3 * (y + 2) ** 2 - 2 * x * y + 4 * x - 6 * y

def grid():
    x = np.linspace(-1.0, 4.0, 220)
    y = np.linspace(-3.0, 2.0, 220)
    xx, yy = np.meshgrid(x, y)
    return xx, yy, original_formula(xx, yy)

def plot_path(points, title, simplexes=None, out=None):
    xx, yy, zz = grid()
    fig, ax = plt.subplots(figsize=(7, 5), dpi=130)
    contour = ax.contour(xx, yy, zz, levels=25, cmap="viridis")
    ax.clabel(contour, inline=True, fontsize=7)
    if simplexes is not None:
        for simplex in simplexes:
            closed = np.vstack([simplex, simplex[0]])
            ax.plot(closed[:, 0], closed[:, 1], color="#4b5563", alpha=0.35, linewidth=1)
    points = np.asarray(points)
    ax.plot(points[:, 0], points[:, 1], marker="o", linewidth=1.8, markersize=3.5, label="траектория")
    ax.scatter([points[0, 0]], [points[0, 1]], color="#2563eb", s=45, label="x0")
    ax.scatter([X_STAR[0]], [X_STAR[1]], color="red", marker="*", s=120, label="x*")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    if out:
        fig.savefig(out, bbox_inches="tight")
    plt.show()

In [ ]:
def nelder_mead(func, x0, step=1.0, tol=1e-4, max_iter=300, alpha=1.0, gamma=2.0, beta=0.5, sigma=0.5):
    x0 = np.asarray(x0, dtype=float)
    simplex = [x0]
    for i in range(len(x0)):
        vertex = x0.copy()
        vertex[i] += step
        simplex.append(vertex)
    simplex = np.array(simplex, dtype=float)
    history = []

    for iteration in range(max_iter + 1):
        values = np.array([func(vertex) for vertex in simplex])
        order = np.argsort(values)
        simplex = simplex[order]
        values = values[order]
        spread = float(np.max(values) - np.min(values))
        history.append({
            "iter": iteration,
            "simplex": simplex.copy(),
            "best_x": simplex[0].copy(),
            "best_f": float(values[0]),
            "spread": spread,
        })
        if spread < tol:
            break

        centroid = simplex[:-1].mean(axis=0)
        worst = simplex[-1]
        reflected = centroid + alpha * (centroid - worst)
        f_reflected = func(reflected)

        if f_reflected < values[0]:
            expanded = centroid + gamma * (reflected - centroid)
            simplex[-1] = expanded if func(expanded) < f_reflected else reflected
        elif f_reflected < values[-2]:
            simplex[-1] = reflected
        else:
            if f_reflected < values[-1]:
                contracted = centroid + beta * (reflected - centroid)
            else:
                contracted = centroid - beta * (centroid - worst)
            if func(contracted) < min(f_reflected, values[-1]):
                simplex[-1] = contracted
            else:
                best = simplex[0].copy()
                simplex = best + sigma * (simplex - best)

    best = history[-1]["best_x"]
    return best, float(func(best)), history


def bisection_min(func, a, b, tol=1e-4):
    while (b - a) > tol:
        mid = (a + b) / 2.0
        left = a + (b - a) / 4.0
        right = b - (b - a) / 4.0
        f_left, f_mid, f_right = func(left), func(mid), func(right)
        if f_left < f_mid:
            b = mid
        elif f_right < f_mid:
            a = mid
        else:
            a, b = left, right
    t = (a + b) / 2.0
    return t, float(func(t))

def exploratory_search(func, base, delta):
    point = base.copy()
    for i in range(len(point)):
        best_value = func(point)
        trial = point.copy()
        trial[i] += delta
        if func(trial) < best_value:
            point = trial
            continue
        trial = point.copy()
        trial[i] -= delta
        if func(trial) < best_value:
            point = trial
    return point

def hooke_jeeves(func, x0, delta=1.0, tol=1e-4, max_iter=300):
    base = np.asarray(x0, dtype=float)
    history = []
    iteration = 0
    while delta > tol and iteration < max_iter:
        explored = exploratory_search(func, base, delta)
        if func(explored) < func(base):
            direction = explored - base
            phi = lambda t: func(explored + t * direction)
            t_star, _ = bisection_min(phi, 0.0, 2.0, tol)
            candidate = explored + t_star * direction
            base = candidate if func(candidate) < func(base) else explored
        else:
            delta *= 0.5
        history.append({"iter": iteration, "x": base.copy(), "f": float(func(base)), "delta": float(delta)})
        iteration += 1
    return base, float(func(base)), history


def steepest_descent(func, grad_func, q_matrix, x0, tol=1e-4, max_iter=300):
    x = np.asarray(x0, dtype=float)
    history = []
    for iteration in range(max_iter + 1):
        g = grad_func(x)
        norm_g = float(np.linalg.norm(g))
        history.append({"iter": iteration, "x": x.copy(), "f": float(func(x)), "grad_norm": norm_g})
        if norm_g < tol:
            break
        alpha = float((g @ g) / (g @ q_matrix @ g))
        x = x - alpha * g
    return x, float(func(x)), history

def fletcher_reeves(func, grad_func, q_matrix, x0, tol=1e-4, max_iter=300):
    x = np.asarray(x0, dtype=float)
    g = grad_func(x)
    direction = -g
    history = []
    for iteration in range(max_iter + 1):
        norm_g = float(np.linalg.norm(g))
        history.append({"iter": iteration, "x": x.copy(), "f": float(func(x)), "grad_norm": norm_g})
        if norm_g < tol:
            break
        alpha = float(-(g @ direction) / (direction @ q_matrix @ direction))
        x_next = x + alpha * direction
        g_next = grad_func(x_next)
        beta = float((g_next @ g_next) / (g @ g))
        direction = -g_next + beta * direction
        x, g = x_next, g_next
    return x, float(func(x)), history

In [ ]:
nm_x, nm_f, nm_h = nelder_mead(f, X0, tol=EPS)
hj_x, hj_f, hj_h = hooke_jeeves(f, X0, tol=EPS)
sd_x, sd_f, sd_h = steepest_descent(f, grad, Q, X0, tol=EPS)
fr_x, fr_f, fr_h = fletcher_reeves(f, grad, Q, X0, tol=EPS)

compare = pd.DataFrame([
    ["Нелдер-Мид", nm_h[-1]["iter"], nm_x[0], nm_x[1], nm_f, np.linalg.norm(nm_x - X_STAR)],
    ["Хук-Дживс", len(hj_h), hj_x[0], hj_x[1], hj_f, np.linalg.norm(hj_x - X_STAR)],
    ["Наискорейший спуск", sd_h[-1]["iter"], sd_x[0], sd_x[1], sd_f, np.linalg.norm(sd_x - X_STAR)],
    ["Флетчер-Ривс", fr_h[-1]["iter"], fr_x[0], fr_x[1], fr_f, np.linalg.norm(fr_x - X_STAR)],
], columns=["Метод", "Итерации", "x", "y", "f", "||x-x*||"])
compare.to_csv("tables/tab_5_compare.csv", index=False)
display(compare)

series = {
    "Нелдер-Мид": [item["best_f"] for item in nm_h],
    "Хук-Дживс": [item["f"] for item in hj_h],
    "Наискорейший спуск": [item["f"] for item in sd_h],
    "Флетчер-Ривс": [item["f"] for item in fr_h],
}
fig, ax = plt.subplots(figsize=(7, 5), dpi=130)
for name, values in series.items():
    err = np.maximum(np.array(values) - F_STAR, 1e-12)
    ax.semilogy(range(len(values)), err, marker="o", markersize=3, linewidth=1.5, label=name)
ax.set_xlabel("Номер итерации")
ax.set_ylabel("f(x_k) - f*")
ax.set_title("Сравнение сходимости")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig("figures/fig_5_convergence.png", bbox_inches="tight")
plt.show()